In [9]:
%pip install pandas scikit-learn langchain-text-splitters langchain-huggingface langchain-community faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- STEP 1: LOAD AND SAMPLE DATA ---
print("Loading data...")
df = pd.read_csv('../data/processed/filtered_complaints.csv')

# Stratified sampling to ensure all 4 products are represented
sample_size = 15000
df_sample, _ = train_test_split(
    df, 
    train_size=sample_size/len(df), 
    stratify=df['Product_Category'], 
    random_state=42
)
print(f"Sample created: {len(df_sample)} complaints.")

# --- STEP 2: CHUNK THE TEXT ---
print("Chunking narratives...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50
)

chunks = []
metadatas = []

for _, row in df_sample.iterrows():
    # Split the cleaned narrative into small pieces
    texts = text_splitter.split_text(str(row['cleaned_narrative']))
    
    for i, text in enumerate(texts):
        chunks.append(text)
        metadatas.append({
            "complaint_id": row['Complaint ID'],
            "product": row['Product_Category'],
            "issue": row['Issue']
        })

print(f"Created {len(chunks)} text chunks.")

# --- STEP 3: LOAD LOCAL EMBEDDING MODEL ---
local_model_path = "../models/all-MiniLM-L6-v2/"

print("Loading local embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name=local_model_path,
    model_kwargs={'device': 'cpu'}
)

# --- STEP 4: BUILD AND SAVE VECTOR STORE ---
print("Generating vectors (this may take a minute)...")
vector_db = FAISS.from_texts(chunks, embeddings, metadatas=metadatas)

# Save the index to disk
if not os.path.exists('../vector_store'):
    os.makedirs('../vector_store')

vector_db.save_local("../vector_store/complaints_index")
print("✅ Task 2 Complete: Vector store saved in '../vector_store/complaints_index'")

# --- STEP 5: FINAL TEST SEARCH ---
print("\n--- Testing Search Capacity ---")
query = "Why are people unhappy with credit card fees?"
results = vector_db.similarity_search(query, k=2)

for i, res in enumerate(results):
    print(f"\nResult {i+1}:")
    print(f"Product: {res.metadata['product']}")
    print(f"Snippet: {res.page_content[:150]}...")

Loading data...
Sample created: 15000 complaints.
Chunking narratives...
Created 46931 text chunks.
Loading local embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9295.41it/s]

Generating vectors (this may take a minute)...


✅ Task 2 Complete: Vector store saved in '../vector_store/complaints_index'

--- Testing Search Capacity ---

Result 1:
Product: Savings Account
Snippet: ive been with     for over 20 years they really charge me left and right from mortgage account to credit card to debit card extra fee never receive my...

Result 2:
Product: Credit Card
Snippet: in my opinion this is borderlining predatory lending for such a high fee to be charged repeatedly without not a simple activity year after year what i...
